In [2]:
#Import models and libraries
from DT.DecisionTree import DecisionTree
from sklearn.tree import DecisionTreeClassifier
#from GNB.gaussian_naive_bayes import GNB
#from LogisticRegresssion.LogisticRegression import MultiLogReg
from tensorflow import keras
from SVM.linear_svm import LinearSVMScartch
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report , f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from Kfolds import run_kfold
import itertools
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from diagnosis_curves import analysis_curves



In [3]:
#Download Data
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()


In [4]:
# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

In [6]:
X_train.shape

(60000, 28, 28)

In [6]:
import numpy as np
from skimage.feature import hog

def extract_hog_features(X):
    features = []

    for img in X:
        img_2d = img.reshape(28, 28)

        hog_features = hog(
            img_2d,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            feature_vector=True
        )

        features.append(hog_features)

    return np.array(features)

In [7]:
X_train_HOG = extract_hog_features(X_train)
X_test_HOG = extract_hog_features(X_test)

In [ ]:
pca = PCA(n_components=50)  

X_train_hog_pca = pca.fit_transform(X_train_HOG)
X_test_hog_pca = pca.transform(X_test_HOG)

In [ ]:
print(X_test_hog_pca.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_hog_pca, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

In [ ]:


def manual_grid_search_DT(X, y, param_grid, k=3):

    keys = list(param_grid.keys())
    values = list(param_grid.values())

    best_score = -1
    best_params = None
    results = []

    for combo in itertools.product(*values):

        params = dict(zip(keys, combo))

        print("\nTesting:", params)

        kf = KFold(n_splits=k, shuffle=True, random_state=42)

        fold_scores = []

        for train_idx, val_idx in kf.split(X):

            X_train_fold = X[train_idx]
            X_val_fold = X[val_idx]

            y_train_fold = y[train_idx]
            y_val_fold = y[val_idx]

            model = DecisionTree(
                maxDepth=params["maxDepth"],
                minSamplesSplit=params["minSamplesSplit"],
                minSampleLeafs=params["minSampleLeafs"],
                criterion=params["criterion"],
                maxFeatures=params["maxFeatures"]
            )

            model.fit(X_train_fold, y_train_fold)

            preds = model.predict(X_val_fold)

            score = f1_score(y_val_fold, preds, average="macro")

            fold_scores.append(score)

        avg_score = sum(fold_scores) / len(fold_scores)

        print("Average score:", avg_score)

        results.append((params, avg_score))

        if avg_score > best_score:
            best_score = avg_score
            best_params = params

    print("\nBest params:", best_params)
    print("Best score:", best_score)

    return best_params, best_score, results

In [5]:
#DT hyper parameter tuning
param_grid = {
    "maxDepth": [8, 12, 15],
    "minSamplesSplit": [5, 10, 20],
    "minSampleLeafs": [1, 5, 10],
    "criterion": ["gini", "entropy"],
    "maxFeatures": ["sqrt", "log2"]
}
best_params, _, _ = manual_grid_search_DT(X_train , y_train , param_grid)



NameError: name 'manual_grid_search_DT' is not defined

In [ ]:
#Decision Tree results
#Best params =>{'maxDepth': 15, 'minSamplesSplit': 20, 'minSampleLeafs': 1, 'criterion': 'entropy', 'maxFeatures': 'sqrt'}
best_params['maxFeatures'] = None
def train_DT(X, y):
    dt = DecisionTree(**best_params)
    dt.fit(X, y)
    return dt
def predict_DT(model, X):
    return model.predict(X)

#Validation
print("Validation results")
# run_kfold(X_train , y_train , train_DT , predict_DT ,k=3, binary=0)

dt = DecisionTree(**best_params)


dt.fit(X_train , y_train)
predictions = dt.predict(X_test_hog_pca)
print(classification_report(
    y_test, predictions,
))



In [ ]:


# Train
gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

# Tune weights
best_weight = None
best_score = -1

def train_gnb(X, y):
    gnb = GNB()
    gnb.gaussian_naive_train(X, y)
    return gnb


def predict_gnb(model, X):
    return model.predict(
        X)



print("\n=== K-FOLD VALIDATION ===")
run_kfold(X_train, y_train, train_gnb, predict_gnb, k=5 , binary=0)



gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

predictions = gnb.predict(
    X_test_hog_pca, 
)

print(classification_report(
    y_test,
    predictions,
))

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

print("Starting Learning Curve evaluations...\n")
def train_multilogreg(X_subset, y_subset):
    model = MultiLogReg(
        max_iterations=100, 
        learning_rate=0.1, 
        reg_eqn='L2',       
        reg_param=0.1,      
        random_state=42
    )
    model.fit(X_subset, y_subset)
    return model

def predict_multilogreg(model, X):
    return model.predict(X)

def metric_acc(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def metric_f1_macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')


print("--- Running Evaluation 1/2: ACCURACY ---")
subset_sizes_acc, train_trends_acc, val_trends_acc = analysis_curves(
    X_train=X_train, 
    y_train=y_train, 
    X_val=X_val, 
    y_val=y_val, 
    train_fn=train_multilogreg, 
    predict_fn=predict_multilogreg, 
    metric_fn=metric_acc, 
    num_steps=8,
    title="MultiLogReg Learning Curve: Accuracy (HOG + PCA)"
)


print("\n--- Running Evaluation 2/2: MACRO F1-SCORE ---")
subset_sizes_f1, train_trends_f1, val_trends_f1 = analysis_curves(
    X_train=X_train, 
    y_train=y_train, 
    X_val=X_val, 
    y_val=y_val, 
    train_fn=train_multilogreg, 
    predict_fn=predict_multilogreg, 
    metric_fn=metric_f1_macro, 
    num_steps=8,
    title="MultiLogReg Learning Curve: Macro F1-Score (HOG + PCA)"
)

In [8]:

# ============================================================
# MultiClass SVM — Setup: imports, grid search, all 3 splits
# ============================================================
from SVM.multiclass_svm import MultiClassSVMScratch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

RANDOM_STATE = 42

# Reload original y_train (60000) in case it was overwritten by earlier split cells
(_, y_train_orig), (_, _) = keras.datasets.mnist.load_data()

def manual_grid_search_SVM(X_tr, y_tr, X_v, y_v, label=''):
    C_values  = [0.1, 1.0, 10.0]
    lr_values = [0.001, 0.0005, 0.0001]
    print(f"{'C':>6}  {'LR':>8}  {'Val Acc':>9}")
    print("-" * 30)
    results = []
    for C in C_values:
        for lr in lr_values:
            model = MultiClassSVMScratch(C=C, learning_rate=lr,
                                         n_epochs=30, random_state=RANDOM_STATE)
            model.fit(X_tr, y_tr)
            acc = accuracy_score(y_v, model.predict(X_v))
            results.append((C, lr, acc))
            print(f"{C:>6}  {lr:>8}  {acc:>9.4f}")
    best = max(results, key=lambda x: x[2])
    print(f"\nBest: C={best[0]}, lr={best[1]}, Val Acc={best[2]:.4f}")
    return best[0], best[1]

# ---------- Split 1: HOG only ----------
scaler_hog = StandardScaler()
X_hog_s      = scaler_hog.fit_transform(X_train_HOG)
X_test_hog_s = scaler_hog.transform(X_test_HOG)

X_tr_hog, X_v_hog, y_tr_hog, y_v_hog = train_test_split(
    X_hog_s, y_train_orig, test_size=0.2, stratify=y_train_orig, random_state=RANDOM_STATE)

# ---------- Split 2: PCA only (raw pixels) ----------
X_flat      = X_train.reshape(X_train.shape[0], -1)   # (60000, 784)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

scaler_raw   = StandardScaler()
X_flat_s     = scaler_raw.fit_transform(X_flat)
X_test_flat_s = scaler_raw.transform(X_test_flat)

pca_only      = PCA(n_components=50, random_state=RANDOM_STATE)
X_pca         = pca_only.fit_transform(X_flat_s)
X_test_pca    = pca_only.transform(X_test_flat_s)

X_tr_pca, X_v_pca, y_tr_pca, y_v_pca = train_test_split(
    X_pca, y_train_orig, test_size=0.2, stratify=y_train_orig, random_state=RANDOM_STATE)

# ---------- Split 3: HOG + PCA ----------
pca_hog       = PCA(n_components=50, random_state=RANDOM_STATE)
X_hog_pca     = pca_hog.fit_transform(X_hog_s)
X_test_hog_pca_svm = pca_hog.transform(X_test_hog_s)

X_tr_hog_pca, X_v_hog_pca, y_tr_hog_pca, y_v_hog_pca = train_test_split(
    X_hog_pca, y_train_orig, test_size=0.2, stratify=y_train_orig, random_state=RANDOM_STATE)

print(f"HOG only  — Train: {X_tr_hog.shape},     Test: {X_test_hog_s.shape}")
print(f"PCA only  — Train: {X_tr_pca.shape},      Test: {X_test_pca.shape}")
print(f"HOG+PCA   — Train: {X_tr_hog_pca.shape},  Test: {X_test_hog_pca_svm.shape}")


HOG only  — Train: (48000, 1296),     Test: (10000, 1296)
PCA only  — Train: (48000, 50),      Test: (10000, 50)
HOG+PCA   — Train: (48000, 50),  Test: (10000, 50)


In [9]:

# ============================================================
# MultiClass SVM — HOG only
# ============================================================
print("=== MultiClass SVM: HOG only ===\n")
best_C, best_lr = manual_grid_search_SVM(X_tr_hog, y_tr_hog, X_v_hog, y_v_hog)

svm_hog = MultiClassSVMScratch(C=best_C, learning_rate=best_lr,
                                n_epochs=50, random_state=RANDOM_STATE)
svm_hog.fit(X_tr_hog, y_tr_hog)

preds = svm_hog.predict(X_test_hog_s)
print(f"\nBest params: C={best_C}, lr={best_lr}")
print(classification_report(y_test, preds))
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")


=== MultiClass SVM: HOG only ===

     C        LR    Val Acc
------------------------------
   0.1     0.001     0.9730
   0.1    0.0005     0.9740
   0.1    0.0001     0.9733
   1.0     0.001     0.9668
   1.0    0.0005     0.9701
   1.0    0.0001     0.9766
  10.0     0.001     0.9584
  10.0    0.0005     0.9646
  10.0    0.0001     0.9719

Best: C=1.0, lr=0.0001, Val Acc=0.9766

Best params: C=1.0, lr=0.0001
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       0.99      0.99      0.99      1135
           2       0.98      0.97      0.98      1032
           3       0.96      0.98      0.97      1010
           4       0.98      0.98      0.98       982
           5       0.98      0.98      0.98       892
           6       0.99      0.98      0.98       958
           7       0.98      0.97      0.98      1028
           8       0.97      0.97      0.97       974
           9       0.97      0.97      0.97

In [10]:

# ============================================================
# MultiClass SVM — PCA only
# ============================================================
print("=== MultiClass SVM: PCA only ===\n")
best_C, best_lr = manual_grid_search_SVM(X_tr_pca, y_tr_pca, X_v_pca, y_v_pca)

svm_pca = MultiClassSVMScratch(C=best_C, learning_rate=best_lr,
                                n_epochs=50, random_state=RANDOM_STATE)
svm_pca.fit(X_tr_pca, y_tr_pca)

preds = svm_pca.predict(X_test_pca)
print(f"\nBest params: C={best_C}, lr={best_lr}")
print(classification_report(y_test, preds))
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")


=== MultiClass SVM: PCA only ===

     C        LR    Val Acc
------------------------------
   0.1     0.001     0.8778
   0.1    0.0005     0.8795
   0.1    0.0001     0.8762
   1.0     0.001     0.8548
   1.0    0.0005     0.8710
   1.0    0.0001     0.8834
  10.0     0.001     0.8013
  10.0    0.0005     0.8448
  10.0    0.0001     0.8700

Best: C=1.0, lr=0.0001, Val Acc=0.8834

Best params: C=1.0, lr=0.0001
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       980
           1       0.97      0.96      0.96      1135
           2       0.90      0.84      0.87      1032
           3       0.86      0.87      0.87      1010
           4       0.87      0.91      0.89       982
           5       0.82      0.83      0.83       892
           6       0.93      0.93      0.93       958
           7       0.90      0.90      0.90      1028
           8       0.81      0.85      0.83       974
           9       0.87      0.86      0.86

In [11]:

# ============================================================
# MultiClass SVM — HOG + PCA
# ============================================================
print("=== MultiClass SVM: HOG + PCA ===\n")
best_C, best_lr = manual_grid_search_SVM(X_tr_hog_pca, y_tr_hog_pca, X_v_hog_pca, y_v_hog_pca)

svm_hog_pca = MultiClassSVMScratch(C=best_C, learning_rate=best_lr,
                                    n_epochs=50, random_state=RANDOM_STATE)
svm_hog_pca.fit(X_tr_hog_pca, y_tr_hog_pca)

preds = svm_hog_pca.predict(X_test_hog_pca_svm)
print(f"\nBest params: C={best_C}, lr={best_lr}")
print(classification_report(y_test, preds))
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")


=== MultiClass SVM: HOG + PCA ===

     C        LR    Val Acc
------------------------------
   0.1     0.001     0.9543
   0.1    0.0005     0.9551
   0.1    0.0001     0.9537
   1.0     0.001     0.9471
   1.0    0.0005     0.9523
   1.0    0.0001     0.9567
  10.0     0.001     0.9360
  10.0    0.0005     0.9378
  10.0    0.0001     0.9497

Best: C=1.0, lr=0.0001, Val Acc=0.9567

Best params: C=1.0, lr=0.0001
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       980
           1       0.98      0.98      0.98      1135
           2       0.97      0.94      0.95      1032
           3       0.95      0.97      0.96      1010
           4       0.97      0.97      0.97       982
           5       0.97      0.95      0.96       892
           6       0.98      0.97      0.97       958
           7       0.95      0.95      0.95      1028
           8       0.91      0.95      0.93       974
           9       0.96      0.94      0.9